# 有状态聊天机器人（Stateful Chatbot）

## 练习目标（理念）

做一个**带记忆**的多轮对话：把 `system` / `user` / `assistant` 消息不断追加到同一个 `messages` 列表里，再交给模型。这样模型能看到上文，而不是每次都「失忆」。

## 和第 2 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多轮 messages | `messages` 列表在递归 `do_chat` 中持续增长 |
| 流式输出 | `stream=True` + `display` / `update_display` 刷新 Markdown |
| 本地 Ollama | OpenAI 兼容端点 `http://localhost:11434/v1`，模型 `llama3.2` |
| System Prompt | 简短助手人设 + Markdown 回复约束 |

## 怎么跑

1. 先启动本机 Ollama，并确保已拉取 `llama3.2`
2. 从上到下运行；「正式对话」格会用 `input()` 交互提问
3. 想继续就再输入；直接回车结束。最后可跑「调试」格把 `messages` 打成 Markdown 表


In [ ]:
# ========== 导入：笔记本展示 + OpenAI 兼容客户端 ==========

# 从 IPython.display 导入：Markdown 渲染、首次 display、以及按 display_id 增量 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI：后面用同一套 SDK 调本地 Ollama（OpenAI 兼容 /v1）
from openai import OpenAI


### GPT 设置（可选，当前整段已注释）

下面代码格是**云端 OpenAI** 路径的草稿：整段保持注释，本笔记本实际用的是后面的 **Ollama** 客户端。若要改用 GPT，需取消注释并注释掉 Ollama 那格。


In [ ]:
# ========== 可选：OpenAI 云端配置（整段保持注释，不改变当前可运行路径）==========
# 说明：原先用 # 【注】 前缀；现改为普通 # 注释 + 中文旁注。逻辑仍不执行。

# 导入 os：读环境变量里的 API Key
# import os
# 从 dotenv 导入 load_dotenv：把 .env 读进环境变量
# from dotenv import load_dotenv

# 加载 .env；override=True 覆盖已有同名变量
# load_dotenv(override=True)
# 读取 OPENAI_API_KEY（常见官方环境变量名）
# api_key = os.getenv('OPENAI_API_KEY')

# 三级体检：有没有 key / 是否像 sk-proj- / 是否首尾空白
# if not api_key:
#     print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# elif not api_key.startswith("sk-proj-"):
#     print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# elif api_key.strip() != api_key:
#     print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# else:
#     print("API key found and looks good so far!")

# 云端模型名示例（保持注释，避免覆盖下面 Ollama 的 MODEL）
# MODEL = "gpt-5-nano"
#
# 默认从环境变量创建 OpenAI 客户端（需上面 key 体检通过）
# my_ai = OpenAI()


### Ollama 设置（本笔记本实际启用）

用 OpenAI 兼容 SDK 指向本机 Ollama：`base_url` + 任意占位 `api_key`（Ollama 本地通常不校验密钥内容）。


In [ ]:
# ========== Ollama：OpenAI 兼容本地客户端 ==========

# Ollama 的 OpenAI 兼容根地址（注意带 /v1）
BASE_URL = "http://localhost:11434/v1"
# 本地占位密钥：SDK 要求传 api_key 字段，Ollama 一般不校验具体值
API_KEY = "ollama"
# 本地模型名：须与 ollama list / ollama pull 的名字一致
MODEL = "llama3.2"

# 创建客户端：后续所有 chat.completions 都走这个 my_ai
my_ai = OpenAI(base_url=BASE_URL, api_key=API_KEY)


#### 其他设置（可选模型名草稿）


In [ ]:
# ========== 可选：换一个更小的本地模型（保持注释）==========
# 若本机已 pull deepseek-r1:1.5b，可取消下一行注释，并注意不要与上面 MODEL 冲突
# MODEL = "deepseek-r1:1.5b"


### 易记提示词（趣味人设草稿，当前未启用）

下面是一段「一本正经胡说八道」的 system prompt 草稿，整段注释；正式对话用的是后文「系统」格里的 `SYSTEM_PROMPT`。


In [ ]:
# ========== 趣味 System Prompt 草稿（整段保持注释，不参与当前运行）==========
# system prompt / thinking prompt 字符串必须保留英文原样——改译会改变模型行为

# SYSTEM_PROMPT = (
#     "You are a helpless assistant that doesn't know anything. "
#     "You make up answers as you go along because you are afraid of being judged. "
#     "You answer with absolute confidence and a serious tone, but most of the information you provide is wrong or made up. "
#     "You are having a conversation with me so your answers shouldn't be too long. "
#     "Reply in markdown. Do not wrap the markdown in a code block - respond just with the markdown."
# )

# 可选：要求模型先写思考过程再给答案（当前未拼进 messages）
# THINKING_PROMPT = "First describe your thought process then give the answer"


## 系统 / 用户提示词

下面先定义「用 Markdown 回复」的公共片段，再拼进正式的 `SYSTEM_PROMPT`。


### 模板：Markdown 回复约束


In [ ]:
# ========== Markdown 回复约束（拼进 system 用）==========
# 要求模型直接输出 Markdown，不要再包一层代码围栏
MARKDOWN_PROMPT = "Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."


### 系统：正式人设


In [ ]:
# ========== 正式 SYSTEM_PROMPT：简短、认真、直给 + Markdown 约束 ==========
# 字符串内容保留英文：这是发给模型的指令，改译会改变回答风格
SYSTEM_PROMPT = (
    "You are a basic assistant. "
    "You solve simple tasks. "
    "Your answers are serious and straight to the point. "
    + MARKDOWN_PROMPT
)


## 辅助函数

流式展示与多轮递归聊天：核心是「边收边刷新」+「把 assistant/user 追加进 messages」。


In [ ]:
# ========== 流式展示：把 stream 增量拼成完整回复并刷新 Markdown ==========

def display_stream(stream):
    # response：累计目前已收到的全部文本
    response = ""
    # 先放一个空的 Markdown，拿到 display_id，后面才能原地更新而不是刷出很多格
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式 chunk；每个 chunk 里可能有一小段 delta.content
    for chunk in stream:
        # or ''：有的 chunk 没有 content（例如只有 role），避免把 None 拼进去
        response += chunk.choices[0].delta.content or ''
        # 用同一个 display_id 更新笔记本中的那一块 Markdown
        update_display(Markdown(response), display_id=display_handle.display_id)
    # 返回完整字符串，方便外层 append 进 messages 当 assistant 内容
    return response


In [ ]:
# ========== 多轮聊天：发问 → 流式回答 → 追加 messages → 可选递归继续 ==========

def do_chat(user_prompt, messages):
    # 打印当前用户问题，方便在输出区对照
    print("Q:", user_prompt)

    print("Answering...")

    # 调用本地（或兼容）Chat Completions；stream=True 边生成边返回
    response = my_ai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    # 边显示边拿到完整回复文本
    full_response = display_stream(response)
    # 把 assistant 回合写进历史，实现「有状态」
    messages.append({"role": "assistant", "content": full_response})

    # 交互：有输入就继续；空回车结束递归
    user_prompt = input("Wanna go further? Enter your prompt, otherwise just press enter: ")

    if user_prompt:
        # 新的 user 消息追加后再递归调用自己
        messages.append({"role": "user", "content": user_prompt})
        print("Answering...")
        do_chat(user_prompt, messages)


## 正式对话

从 `input()` 读第一句，初始化带 system 的 `messages`，然后进入 `do_chat`。


In [ ]:
# ========== 入口：首轮 user + system，启动有状态对话 ==========

# 交互读入第一句用户问题
user_prompt = input("Enter your first prompt: ")
# messages 初始结构：system 定规矩，user 放第一问
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]
# 进入递归多轮；结束后 messages 里会留下完整对话轨迹
do_chat(user_prompt, messages)


### 调试：把 messages 打成 Markdown 表

跑完对话后执行：检查每条消息的 `role` / `content` 是否按预期累积。


In [ ]:
# ========== 调试：用 Markdown 表格可视化 messages 历史 ==========

# 表头取自第一条消息的键（通常是 role、content）
headers = messages[0].keys()

# 拼 Markdown 表头行
table = "| " + " | ".join(headers) + " |\n"
# 分隔行：每个列一个 ---
table += "| " + " | ".join("---" for _ in headers) + " |\n"

# 逐条消息追加一行；缺键用空字符串
for row in messages:
    table += "| " + " | ".join(str(row.get(h, "")) for h in headers) + " |\n"

# 在笔记本里渲染成表格，便于检查「有状态」是否生效
display(Markdown(table))
